In [1]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, f1_score

# Load cleaned dataset (not raw)
df = pd.read_csv("../data/processed/cleaned_survey.csv")

# Drop Timestamp
if 'Timestamp' in df.columns:
    df = df.drop('Timestamp', axis=1)

# Drop missing target
df = df.dropna(subset=['treatment'])

# Convert target
df['treatment'] = df['treatment'].map({'No': 0, 'Yes': 1})

# Fill missing values
for col in df.columns:
    if col != 'treatment':
        df[col] = df[col].fillna(df[col].mode()[0])

# ✅ Add engineered features
df['has_support'] = (
    (df['benefits'] == 'Yes').astype(int) +
    (df['wellness_program'] == 'Yes').astype(int) +
    (df['seek_help'] == 'Yes').astype(int)
)

df['disclosure_comfort'] = (
    (df['coworkers'] == 'Yes').astype(int) +
    (df['supervisor'] == 'Yes').astype(int)
)

df['age_group'] = pd.cut(
    df['Age'],
    bins=[15, 25, 35, 45, 55, 75],
    labels=['15-25', '26-35', '36-45', '46-55', '55+']
).astype(str)

# Split X and y
X = df.drop('treatment', axis=1)
y = df['treatment']

# One-hot encoding
X = pd.get_dummies(X, drop_first=True)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Data ready")

Data ready


Logistic Regression:

In [2]:
lr = LogisticRegression(max_iter=1000)

lr.fit(X_train, y_train)

y_pred = lr.predict(X_test)

print(classification_report(y_test, y_pred))
print("Logistic Regression F1:", f1_score(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.75      0.83      0.79       124
           1       0.81      0.72      0.77       127

    accuracy                           0.78       251
   macro avg       0.78      0.78      0.78       251
weighted avg       0.78      0.78      0.78       251

Logistic Regression F1: 0.7666666666666667


Random Forest:

In [3]:
rf = RandomForestClassifier(
    n_estimators=200,
    class_weight='balanced',
    random_state=42
)

rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)

print(classification_report(y_test, y_pred_rf))
print("Random Forest F1:", f1_score(y_test, y_pred_rf))

              precision    recall  f1-score   support

           0       0.77      0.77      0.77       124
           1       0.77      0.77      0.77       127

    accuracy                           0.77       251
   macro avg       0.77      0.77      0.77       251
weighted avg       0.77      0.77      0.77       251

Random Forest F1: 0.7716535433070866


XGBoost

In [4]:
from xgboost import XGBClassifier

xgb = XGBClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=4,
    random_state=42,
    eval_metric='logloss'
)

xgb.fit(X_train, y_train)

y_pred_xgb = xgb.predict(X_test)

print(classification_report(y_test, y_pred_xgb))
print("XGBoost F1:", f1_score(y_test, y_pred_xgb))

              precision    recall  f1-score   support

           0       0.75      0.78      0.77       124
           1       0.78      0.75      0.76       127

    accuracy                           0.76       251
   macro avg       0.77      0.77      0.76       251
weighted avg       0.77      0.76      0.76       251

XGBoost F1: 0.7630522088353414


comparison table:

In [5]:
results = pd.DataFrame({
    'Model': ['Logistic Regression', 'Random Forest', 'XGBoost'],
    'F1 Score': [
        f1_score(y_test, y_pred),
        f1_score(y_test, y_pred_rf),
        f1_score(y_test, y_pred_xgb)
    ]
})

results

,Model,F1 Score
0,Logistic Regression,0.766667
1,Random Forest,0.771654
2,XGBoost,0.763052


In [7]:
import joblib

# Save Random Forest as the best model
joblib.dump(rf, "../models/best_model.pkl")

# Save model columns
joblib.dump(X.columns.tolist(), "../models/model_columns.pkl")

print("Random Forest model saved as best_model.pkl")

Random Forest model saved as best_model.pkl
